# aw_05_b1b2 — Stages B1/B2: Phase-1 general SFT, two arms (Track B, RQ1+RQ2)

**Protocol**: §5.1 B1/B2. Two arms differing ONLY in response provenance:
- **B1**: curated corpus (`data/p1/p1_general_sft.jsonl`)
- **B2**: rejection-sampled corpus — base-model samples gated by ExactAnswerVerifier
  (`scripts/build_p1_rs_data.py`), same prompt set.

Readouts per arm: (1) P1 general held-out accuracy (`run_p1_eval.py`, retention anchor
is the BASE model's score), (2) **transfer probe** — frozen 200-step PlayWorld SFT
(`probe_playworld_sft.yaml`) evaluated on the frozen suites; probe eval-ID goal-valid
accuracy is the Phase-1 champion primary metric (§6).

**Repos**: B1 → `m97j/aw-runs-b1`, B2 → `m97j/aw-runs-b2`; probe runs get their own
repos `m97j/aw-runs-b1-probe`, `m97j/aw-runs-b2-probe` (fetch_run reads repo-root
artifacts).

**v0.6.2 additions (x09 termination audit, hypothesis T)**:
- `x09a_prereq` — lazy-materialize P1 data (local → HF dataset repo → rebuild guidance).
- `x09b_label_audit` — **GPU-free GATE before training**: rebuilds labels through the
  REAL trainer factory (tiny random model) and checks whether the terminal
  `<|im_end|>`/EOS label is masked (-100) under `pad_token == eos_token`. If
  CONFIRMED, do NOT train — fix `models/builder.py` padding policy first.
- `c_b12_p1_eval` now passes `--dump-predictions` (per-row jsonl, synced to HF via
  the same `p1_eval/` upload) feeding `x09d_drift_audit`.
- `x09c_run_audit` — audits probe-eval run artifacts (`truncated_outputs`, verdict
  reason codes, runaway-tail heuristic).
- `x09d_drift_audit` — GSM8K answer-extraction drift using the verifier's own
  `extract_final_answer` (marker-anchored vs naive last-number vs gold).
- `x11_adapter_integrity` — after x10 returned bit-identical B1==B2 logits with
  near-uniform distributions, audits adapter dirs (sha256, tensor diff, all-zero
  lora_B, forward logit distance) to confirm/refute hypothesis V (adapter
  identity/integrity failure).
- `x13_trained_ids_nll` — replays adapters on the trainer-factory processed
  input_ids (exact trained rendering) to separate rendering mismatch (hyp. W)
  from a degenerate checkpoint; run for base/b1/b2.
- `x12_attested_probe` — single-process re-probe with runtime LoRA attestation
  (disk sha256 vs live-tensor sha256 vs probe outputs) resolving the x10/x11
  conflict; x09c now dedupes duplicated trace rows by (suite, id).
- `x10_stop_logit_probe` — after x09b rejected hypothesis T, discriminates
  U1 (stop-logit under-training) vs U2 (train/eval rendering mismatch) via a
  teacher-forcing probe of P(<|im_end|>) at the gold stop position, base vs adapter,
  train-rendering vs eval-rendering.

**v0.6.3**: x13 confirmed the terminal-stop mechanism (hypothesis-K analogue);
B1'/B2' retrain cells (`g_b12_v2_*`) use `adapter.modules_to_save=[lm_head,
embed_tokens]` per protocol amendment v1.2, with an x13-based fix gate before
any further eval spend.

Session-resume policy everywhere: local reuse → HF fetch → loud failure with the
exact prerequisite stage to run. PlayWorld data/suites are cheap → always rebuilt.

**v0.6.4**: fix gate PASSED for BOTH v2 arms (b1v2: terminal NLL 0.082 / p 0.922;
b2v2: 0.086 / 0.918, rank 1 everywhere incl. a 732-token sample) — hypothesis K
causally confirmed. The eval cycle (`c_b12v2_p1_eval` → `x09d` → `d_b12v2_probe`
→ `e_b12v2_probe_eval` → `x09c` → `f_b12v2_analysis`) is rewired to the V2
adapters/recipes with v2-suffixed artifacts; v1 B1/B2 runs are retained as the
documented failure baseline. `g_b12_v2_verify` now enforces the p_im_end > 0.5
criterion in code and probes each arm on its OWN corpus.


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


Cloning into 'axiom-world'...
remote: Enumerating objects: 550, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 550 (delta 51), reused 56 (delta 21), pack-reused 451 (from 1)
Receiving objects: 100% (550/550), 243.12 KiB | 2.61 MiB/s, done.
Resolving deltas: 100% (283/283), done.
/content/axiom-world
Obtaining file:///content/axiom-world
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 54.6 MB/s eta 0:00:00
  Building editable for axiom-world (pyproject.toml) ... done
  Cre

In [ ]:
# @title x09a_prereq — materialize x09/eval inputs (local -> HF -> rebuild/fail)
from pathlib import Path

HF_DATASET_REPO = "m97j/axiom-general-posttrain"  # dataset repo used by a_b12_data syncs
P1_PATH_IN_REPO = "p1/v1"  # subdir inside the dataset repo holding p1 files ("" = repo root)

P1_TRAIN   = Path("data/p1/p1_general_sft.jsonl")      # x09b (check C) input
P1_TRAIN_RS = Path("data/p1/p1_general_sft_rs.jsonl")
P1_HOLDOUT = Path("data/p1/p1_general_holdout.jsonl")  # run_p1_eval input
P1_FILES = ("p1_general_sft.jsonl", "p1_general_sft_rs.jsonl", "p1_general_holdout.jsonl", "p1_manifest.json", "p1_rs_manifest.json")


def materialize_p1() -> None:
    if P1_TRAIN.exists() and P1_HOLDOUT.exists() and P1_TRAIN_RS.exists():
        print("[x09a] p1 data: LOCAL reuse")
        return
    try:
        from huggingface_hub import hf_hub_download
        P1_TRAIN.parent.mkdir(parents=True, exist_ok=True)
        for name in P1_FILES:
            rel = f"{P1_PATH_IN_REPO}/{name}" if P1_PATH_IN_REPO else name
            got = hf_hub_download(HF_DATASET_REPO, rel, repo_type="dataset")
            (P1_TRAIN.parent / name).write_bytes(Path(got).read_bytes())
        assert P1_TRAIN.exists() and P1_HOLDOUT.exists()
        print("[x09a] p1 data: fetched from HF dataset repo")
    except Exception as exc:  # noqa: BLE001 — surface guidance, then fail loudly
        raise SystemExit(
            "[x09a] p1 data missing locally AND on HF dataset repo "
            f"({HF_DATASET_REPO}).\n"
            "Run stage a_b12_data first (python scripts/build_p1_data.py ...).\n"
            "WARNING: rebuilding may change the frozen holdout fingerprint — "
            "verify p1_manifest.json holdout_fingerprint against the protocol."
        ) from exc


materialize_p1()

# PlayWorld artifacts are cheap to regenerate -> never persisted, always rebuildable
if not Path("data/eval_suites").exists():
    !python scripts/build_eval_suites.py --episodes-per-suite 300
if not Path("data/training").exists():
    !python scripts/build_training_data.py

p1_general_sft.jsonl:   0%|          | 0.00/5.86M [00:00<?, ?B/s]

p1_general_sft_rs.jsonl:   0%|          | 0.00/7.26M [00:00<?, ?B/s]

p1_general_holdout.jsonl:   0%|          | 0.00/473k [00:00<?, ?B/s]

p1_manifest.json:   0%|          | 0.00/673 [00:00<?, ?B/s]

p1_rs_manifest.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

[x09a] p1 data: fetched from HF dataset repo
{
  "seed": 1042,
  "sft_records": 2000,
  "prompt_records": 2000,
  "unsolvable_dropped": 0,
  "sft_fingerprint": "sha256:3144c43ae76020e908e03fbbc0654c7a6f060cc9703150fc84274a9a213c209a",
  "prompt_fingerprint": "sha256:cc2aef0df4f5efda3db7ccfd43c76e40260935e9b7b55ee5760490e6a4304f14",
  "train_families": [
    "train-fam0",
    "train-fam1",
    "train-fam2",
    "train-fam3",
    "train-fam4"
  ],
  "eval_families_checked": [
    "eval_adversarial-fam0",
    "eval_comp_ood-fam0",
    "eval_comp_ood-fam1",
    "eval_id-fam0",
    "eval_id-fam1",
    "eval_id-fam2",
    "eval_rule_ood-fam0",
    "eval_rule_ood-fam1",
    "eval_template_ood-fam0",
    "eval_template_ood-fam1"
  ]
}


In [ ]:
# @title x09b_label_audit — hypothesis-T GATE via the REAL trainer factory (CPU, no GPU needed)
# Runs BEFORE any (re)training: if the terminal stop-token label is masked in the
# collated batch under pad_id == eos_id, SFT arms cannot learn to terminate and
# further training spend is wasted until models/builder.py padding policy is fixed.
!python scripts/x09_termination_audit.py \
  --sft-config configs/experiments/b1_general_sft.yaml \
  --sft-jsonl data/p1/p1_general_sft.jsonl \
  --n-label-samples 4 \
  --out runs/x09_label_audit.json

import json as _json

_verdict = _json.load(open("runs/x09_label_audit.json"))["label_audit"]["verdict"]
print("\n[x09b VERDICT]", _verdict)
HYPOTHESIS_T_CONFIRMED = _verdict.startswith("HYPOTHESIS-T CONFIRMED")
if HYPOTHESIS_T_CONFIRMED:
    print(
        "[x09b] GATE: SKIP b_b1_train / b_b2_train — fix models/builder.py padding "
        "policy (distinct pad token) + protocol v1.2 amendment, then retrain."
    )

config.json: 100% 729/729 [00:00<00:00, 8.49MB/s]
tokenizer_config.json: 100% 9.68k/9.68k [00:00<00:00, 7.22MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 116MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 279MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 324MB/s]
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Tokenizing train dataset: 100% 4/4 [00:00<00:00, 253.66 examples/s]
Building labels for train dataset: 100% 4/4 [00:00<00:00, 2114.86 examples/s]
Truncating train dataset: 100% 4/4 [00:00<00:00, 2173.78 examples/s]
Dr

In [ ]:
# @title a_b12_data — P1 mixture + held-out + RS corpus (skip pieces already materialized)
!python scripts/build_p1_data.py
# base-model reference on the held-out (retention anchor, run once; left-padding fixed)
!python scripts/run_p1_eval.py \
  --config configs/experiments/b1_general_sft.yaml \
  --holdout data/p1/p1_general_holdout.jsonl \
  --label qwen3-8b-base --output runs/p1_eval_base.json \
  --dump-predictions runs/p1_eval_base_predictions.jsonl \
  --hf-sync-repo m97j/aw-runs-b1
# B2 rejection-sampled corpus (generation-heavy; skip if already on the dataset repo)
!python scripts/build_p1_rs_data.py \
  --config configs/experiments/b1_general_sft.yaml \
  --input data/p1/p1_general_sft.jsonl \
  --output data/p1/p1_general_sft_rs.jsonl \
  --num-candidates 4 --batch-size 64 \
  --hf-sync-repo m97j/axiom-general-posttrain

README.md: 100% 7.93k/7.93k [00:00<00:00, 20.7MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   8% 185k/2.31M [00:00<00:06, 336kB/s]
main/train-00000-of-00001.parquet: downloading bytes:  57% 1.32M/2.31M [00:00<00:00, 2.60MB/s, 52.1kB/s  ]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:00<00:00, 2.55MB/s,  225kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:00<00:00, 2.56MB/s,  226kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:06<00:00, 66.2kB/s, 36.3kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:06<00:00, 66.2kB/s, 36.3kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 981341.07 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 593656.72 examples/s]
README.md: 100% 3.93k/3.93k [00:00<00:00, 27.4MB/s]

algebra/train-00000-of-00001.parqu

In [ ]:
# @title b_b1_train — P1 general SFT (curated arm)
assert not HYPOTHESIS_T_CONFIRMED, "x09b gate: fix padding policy before training"
!python scripts/run_experiment.py \
  --config configs/experiments/b1_general_sft.yaml \
  --hf-sync-repo m97j/aw-runs-b1

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
run_id: 20260803-162417--b1-general-sft--s42--b08a2c
Loading weights: 100% 399/399 [00:01<00:00, 364.35it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Tokenizing train dataset: 100% 6043/6043 [00:01<00:00, 3628.71 examples/s]
Building labels for train dataset: 100% 6043/6043 [00:00<00:00, 13249.78 examples/s]
Truncating train dataset: 100% 6043/6043 [00:00<00:00, 10126.13 examples/s]
Dropping fully masked examples from train dataset: 100% 6043/6043 [00:00<00:00, 12769.94 examples/s]
[transformers] The tokenizer has new PAD/BOS/

In [ ]:
# @title b_b2_train — P1 general SFT (rejection-sampled arm)
assert not HYPOTHESIS_T_CONFIRMED, "x09b gate: fix padding policy before training"
!python scripts/run_experiment.py \
  --config configs/experiments/b2_general_sft_rs.yaml \
  --hf-sync-repo m97j/aw-runs-b2

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
run_id: 20260803-165015--b2-general-sft-rs--s42--10ecce
Loading weights: 100% 399/399 [00:01<00:00, 365.67it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Tokenizing train dataset: 100% 5654/5654 [00:01<00:00, 2917.64 examples/s]
Building labels for train dataset: 100% 5654/5654 [00:00<00:00, 10738.10 examples/s]
Truncating train dataset: 100% 5654/5654 [00:00<00:00, 8028.88 examples/s]
Dropping fully masked examples from train dataset: 100% 5654/5654 [00:00<00:00, 10166.00 examples/s]
[transformers] The tokenizer has new PAD/BO

In [ ]:
# @title c_b12v2_p1_eval — held-out accuracy for both V2 arms (retention readout, + per-row dumps)
# v0.6.4: the v1 B1/B2 adapters are superseded (termination pathology, protocol
# v1.2 §13); the eval cycle now consumes the fix-gate-passing V2 adapters and
# V2 recipes. Labels/outputs carry the v2 suffix so v1 artifacts stay intact.
b1v2_dir  # <- defined in g_b12_v2_verify (run it first in this session)
b2v2_dir

!python scripts/run_p1_eval.py \
  --config configs/experiments/b1_general_sft_v2.yaml \
  --adapter-dir {b1v2_dir} --label b1v2-sft \
  --output runs/p1_eval_b1v2.json \
  --dump-predictions runs/p1_eval_b1v2_predictions.jsonl \
  --hf-sync-repo m97j/aw-runs-b1
!python scripts/run_p1_eval.py \
  --config configs/experiments/b2_general_sft_rs_v2.yaml \
  --adapter-dir {b2v2_dir} --label b2v2-sft-rs \
  --output runs/p1_eval_b2v2.json \
  --dump-predictions runs/p1_eval_b2v2_predictions.jsonl \
  --hf-sync-repo m97j/aw-runs-b2


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Loading weights: 100% 399/399 [00:01<00:00, 338.89it/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
p1-eval(batched): 100% 5/5 [05:14<00:00, 62.85s/it]
{
  "accuracy": {
    "ci95": [
      0.812,
 

In [ ]:
# @title x09d_drift_audit — GSM8K answer-extraction drift per V2 arm (CPU)
# Consumes the per-row dumps from c_b12v2_p1_eval. New session: dumps live under
# p1_eval/ in each run repo (same upload_directory sync as the summary json).
from pathlib import Path

for label, repo in (("b1v2", "m97j/aw-runs-b1"), ("b2v2", "m97j/aw-runs-b2")):
    pred = Path(f"runs/p1_eval_{label}_predictions.jsonl")
    if not pred.exists():
        try:
            from huggingface_hub import hf_hub_download
            got = hf_hub_download(repo, f"p1_eval/{pred.name}")
            pred.parent.mkdir(parents=True, exist_ok=True)
            pred.write_bytes(Path(got).read_bytes())
            print(f"[x09d] {pred.name}: fetched from HF ({repo})")
        except Exception as exc:  # noqa: BLE001
            raise SystemExit(
                f"[x09d] {pred} not found locally or on {repo}. "
                "Re-run c_b12v2_p1_eval with --dump-predictions first (GPU stage)."
            ) from exc
    else:
        print(f"[x09d] {pred.name}: LOCAL reuse")
    !python scripts/x09_termination_audit.py \
      --p1-predictions {pred} --out runs/x09_drift_audit_{label}.json


[x09d] p1_eval_b1v2_predictions.jsonl: LOCAL reuse
{
  "gsm8k_drift_audit": {
    "predictions_file": "runs/p1_eval_b1v2_predictions.jsonl",
    "rows": 500,
    "verifier_pass_rate": 0.844,
    "marker_present_rate": 0.726,
    "marker_vs_lastnum_drift": 0,
    "drift_caused_failures": 0,
    "read_out": "high truncation + LOW drift_caused_failures => collapse is real non-termination/competence, not answer extraction; drift_caused_failures > 0 => part of the GSM8K drop is runaway text confusing extraction (still a termination symptom)."
  }
}
[x09d] p1_eval_b2v2_predictions.jsonl: LOCAL reuse
{
  "gsm8k_drift_audit": {
    "predictions_file": "runs/p1_eval_b2v2_predictions.jsonl",
    "rows": 500,
    "verifier_pass_rate": 0.834,
    "marker_present_rate": 0.084,
    "marker_vs_lastnum_drift": 3,
    "drift_caused_failures": 0,
    "read_out": "high truncation + LOW drift_caused_failures => collapse is real non-termination/competence, not answer extraction; drift_caused_failures > 0 =

In [ ]:
# @title d_b12v2_probe — frozen transfer probe (200 steps) per V2 arm
assert not HYPOTHESIS_T_CONFIRMED, "x09b gate: fix padding policy before training"
import json

b1v2_sha = json.load(open(f"runs/{B1V2_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]
b2v2_sha = json.load(open(f"runs/{B2V2_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]

!python scripts/run_experiment.py \
  --config configs/experiments/probe_playworld_sft.yaml \
  --parent-adapter-dir {b1v2_dir} \
  --override lineage.parent_adapter.repo_id=m97j/aw-runs-b1 \
  --override lineage.parent_adapter.sha256={b1v2_sha} \
  --override experiment_name=probe-playworld-sft-b1v2 \
  --hf-sync-repo m97j/aw-runs-b1-probe

!python scripts/run_experiment.py \
  --config configs/experiments/probe_playworld_sft.yaml \
  --parent-adapter-dir {b2v2_dir} \
  --override lineage.parent_adapter.repo_id=m97j/aw-runs-b2 \
  --override lineage.parent_adapter.sha256={b2v2_sha} \
  --override experiment_name=probe-playworld-sft-b2v2 \
  --hf-sync-repo m97j/aw-runs-b2-probe


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
run_id: 20260810-055532--probe-playworld-sft-b1v2--s42--4f2c4f
Loading weights: 100% 399/399 [00:01<00:00, 340.65it/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
[transformers] warmup_ratio is depr

In [ ]:
# @title e_b12v2_probe_eval — V2 probe adapters on the frozen suites
B1V2_PROBE_RUN = "20260810-055532--probe-playworld-sft-b1v2--s42--4f2c4f"
B2V2_PROBE_RUN = "20260810-060645--probe-playworld-sft-b2v2--s42--e3f2d5"

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1-probe --run-id {B1V2_PROBE_RUN}
b1v2p_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b2-probe --run-id {B2V2_PROBE_RUN}
b2v2p_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b1v2p_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b1-probe
!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b2v2p_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b2-probe


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Loading weights: 100% 399/399 [00:01<00:00, 341.04it/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
generate(batched): 100% 3/3 [00:30<00:00, 10.20s/it]
eval_adversarial: pass_rate={'mean': 0.8133, 

In [ ]:
# @title x09c_run_audit — termination audit of V2 probe-eval run artifacts (CPU)
# Primary regression check for the v1.2 amendment: truncation_rate must drop
# from 100% (v1 pathology) to ~0 now that the stop token is learned.
B1V2_PROBE_EVAL = "20260810-063029--eval-playworld--s42--a7a47a"
B2V2_PROBE_EVAL = "20260810-063502--eval-playworld--s42--cd89fd"

# local -> HF (fetch_run --kind eval verifies against the persisted run)
!python scripts/fetch_run.py --repo m97j/aw-runs-b1-probe --run-id {B1V2_PROBE_EVAL} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-b2-probe --run-id {B2V2_PROBE_EVAL} --kind eval

!python scripts/x09_termination_audit.py \
  --run-dirs runs/{B1V2_PROBE_EVAL} runs/{B2V2_PROBE_EVAL} \
  --out runs/x09_run_audit_v2.json


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0% 0/8 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.28k [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   1% 3.28k/456k [00:00<00:19, 22.8kB/s] 
Reconstructing (incomplete total...):  50% 456k/912k [00:00<00:20, 22.8kB/s] 
Reconstructing (incomplete total...):  67% 912k/1.36M [00:00<00:00, 4.30MB/s]
Reconstructing (incomplete total...): 100% 1.36M/1.36M [00:00<00:00, 4.30MB/s]

Fetching 8 files:  12% 1/8 [00:00<00:02,  3.45it/s]
Reconstructing (incomplete total...):  76% 1.36M/1.80M [00:00<00:00, 4.30MB/s]
Reconstructing (incomplete total...):  76% 1.36M/1.80M [00:00<00:00, 4.30MB/s]
Reconstructing (incomplete total...):  82% 1.80M/2.19M [00:00<00:00, 4.30MB/s]
Fetching 8 files: 100% 8/8 [00:00<00:00, 21.15it/s]
Download complete: 100% 2.19M/2.19M [00:00<00:00, 6.93MB/s]
Reconstruction complete: 100% 2.19M/2.19M [00:00<00:00, 6.93MB/s]             eva

In [ ]:
# @title x10_stop_logit_probe — U1 (under-training) vs U2 (rendering mismatch) discrimination
# Context: x09b REJECTED hypothesis T (terminal <|im_end|> labels LIVE); x09c shows
# runaway_rate ~0.96-0.98 on probe evals. Teacher-forcing probe of P(<|im_end|>) at
# the gold stop position under TRAIN vs EVAL rendering. Base run = reference delta.
B1_RUN_ID_X10 = "20260803-162417--b1-general-sft--s42--b08a2c"
B2_RUN_ID_X10 = "20260803-165015--b2-general-sft-rs--s42--10ecce"

out_b1 = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1_RUN_ID_X10}
out_b2 = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B2_RUN_ID_X10}

b1_dir_x10 = [line for line in out_b1 if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
b2_dir_x10 = [line for line in out_b2 if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

# base-model reference (no adapter)
!python scripts/x10_stop_logit_probe.py \
  --config configs/experiments/b1_general_sft.yaml \
  --sft-jsonl data/p1/p1_general_sft.jsonl \
  --num-samples 8 --eval-rendering \
  --out runs/x10_stop_logit_base.json

# B1 adapter
!python scripts/x10_stop_logit_probe.py \
  --config configs/experiments/b1_general_sft.yaml \
  --adapter-dir {b1_dir_x10} \
  --sft-jsonl data/p1/p1_general_sft.jsonl \
  --num-samples 8 --eval-rendering \
  --out runs/x10_stop_logit_b1.json

# B2 adapter
!python scripts/x10_stop_logit_probe.py \
  --config configs/experiments/b2_general_sft_rs.yaml \
  --adapter-dir {b2_dir_x10} \
  --sft-jsonl data/p1/p1_general_sft.jsonl \
  --num-samples 8 --eval-rendering \
  --out runs/x10_stop_logit_b2.json

import json as _json  # noqa: E402

for tag in ("base", "b1", "b2"):
    rep = _json.load(open(f"runs/x10_stop_logit_{tag}.json"))  # noqa: SIM115
    print(f"[x10:{tag}] mean_p_im_end={rep['mean_p_im_end']}")
    print(f"[x10:{tag}] VERDICT: {rep['verdict']}\n")


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading weights: 100% 399/399 [00:01<00:00, 344.45it/s]
{
  "config": "configs/experiments/b1_general_sft.yaml",
  "adapter_dir": null,
  "arm": "base",
  "opener_seed": "",
  "im_end_id": 151645,
  "samples": [
    {
      "idx": 0,
      "train_rendering": {
        "gold_stop_position": 208,
        "p_im_end": 0.0,
        "rank_im_end": 122532,
        "top_tokens": [
          {
            "token": "Ċ",
            "p": 0.775
          },
          {
            "token": "####",
            "p": 0.0926
          },
          {
            "token": "<|endoftext|>",
            

In [ ]:
# @title x11_adapter_integrity — explain the bit-identical B1==B2 x10 anomaly (CPU; --forward-diff on GPU)
# x10 returned IDENTICAL logit reports for the B1 and B2 adapters (impossible for
# independently trained LoRAs) and near-uniform garbage distributions. x11 hashes
# both adapter dirs, diffs tensors elementwise, checks all-zero lora_B (no-op
# adapter), and (GPU) measures forward logit distance base vs each adapter.
B1_RUN_ID_X11 = "20260803-162417--b1-general-sft--s42--b08a2c"  # <- b1-general-sft run id
B2_RUN_ID_X11 = "20260803-165015--b2-general-sft-rs--s42--10ecce"  # <- b2-general-sft-rs run id

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1_RUN_ID_X11}
b1_dir_x11 = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b2 --run-id {B2_RUN_ID_X11}
b2_dir_x11 = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
print("b1:", b1_dir_x11)
print("b2:", b2_dir_x11)
assert b1_dir_x11 != b2_dir_x11, "fetch_run returned the SAME dir for both runs (wiring bug found)"

!python scripts/x11_adapter_integrity.py \
  --adapter-dirs {b1_dir_x11} {b2_dir_x11} \
  --config configs/experiments/b1_general_sft.yaml \
  --forward-diff \
  --out runs/x11_adapter_integrity.json

import json as _json

rep = _json.load(open("runs/x11_adapter_integrity.json"))
print("\n[x11 pairwise]", [p.get("verdict") for p in rep["pairwise"]])
print("[x11 VERDICT]", rep["verdict"])

b1: runs/20260803-162417--b1-general-sft--s42--b08a2c/artifacts/final_adapter
b2: runs/20260803-165015--b2-general-sft-rs--s42--10ecce/artifacts/final_adapter
config.json: 100% 729/729 [00:00<00:00, 8.52MB/s]
tokenizer_config.json: 100% 9.68k/9.68k [00:00<00:00, 7.21MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 29.6MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 193MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 265MB/s]
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
model.safetensors.index.json: 100% 32.9k/32.9k [00:00<00:00, 197MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetchi

In [ ]:
# @title x12_attested_probe — in-process re-probe with runtime weight attestation
# Resolves the x10 (identical B1==B2 logits) vs x11 (weights differ on disk AND in
# forward pass) conflict: ONE process loads each adapter, hashes the LoRA tensors
# actually resident in the model, and repeats the x10 stop-logit probe. Fresh dirs
# are guaranteed by deleting any stale local copies before fetch_run.
B1_RUN_ID_X12 = "20260803-162417--b1-general-sft--s42--b08a2c"  # <- b1-general-sft run id
B2_RUN_ID_X12 = "20260803-165015--b2-general-sft-rs--s42--10ecce"  # <- b2-general-sft-rs run id

!rm -rf runs/{B1_RUN_ID_X12} runs/{B2_RUN_ID_X12}   # kill stale-dir suspects
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1_RUN_ID_X12}
b1_dir_x12 = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b2 --run-id {B2_RUN_ID_X12}
b2_dir_x12 = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/x12_attested_probe.py \
  --config configs/experiments/b1_general_sft.yaml \
  --adapter-dirs {b1_dir_x12} {b2_dir_x12} \
  --labels b1 b2 \
  --sft-jsonl data/p1/p1_general_sft.jsonl \
  --num-samples 4 \
  --out runs/x12_attested_probe.json

import json as _json

rep = _json.load(open("runs/x12_attested_probe.json"))
for arm in rep["arms"]:
    print(f"[x12:{arm['label']}] disk={arm['disk_safetensors_sha256'][:12]} "
          f"live={str(arm['live_lora_sha256'])[:12]} l2={arm['live_lora_global_l2']}")
print("[x12 comparisons]", rep["comparisons"])
for v in rep["verdict"]:
    print("[x12 VERDICT]", v)

config.json: 100% 729/729 [00:00<00:00, 7.03MB/s]
tokenizer_config.json: 100% 9.68k/9.68k [00:00<00:00, 6.83MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 69.9MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 218MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 318MB/s]
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
model.safetensors.index.json: 100% 32.9k/32.9k [00:00<00:00, 43.7MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/7.96G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0

In [ ]:
# @title x13_trained_ids_nll — NLL on the EXACT trained token ids (per-arm, subprocess-safe)
# NOTE: the previous version drove the three runs with `!python ... {label}` inside a
# Python for-loop; Colab's `!` variable expansion did not re-expand per iteration, so
# all three subprocesses overwrote runs/x13_trained_ids_nll_base.json. This version
# builds each command as a plain Python list and runs it via subprocess (no expansion).
import subprocess

B1_RUN_ID_X13 = "20260807-225109--b1-general-sft-v2--s42--e6e83b"  # reuse b1-general-sft run id
B2_RUN_ID_X13 = "20260807-232301--b2-general-sft-rs-v2--s42--9293e7"  # reuse b2-general-sft-rs run id

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1_RUN_ID_X13}
b1_dir_x13 = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b2 --run-id {B2_RUN_ID_X13}
b2_dir_x13 = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

for label, adp in (("base", None), ("b1", b1_dir_x13), ("b2", b2_dir_x13)):
    cmd = ["python", "scripts/x13_trained_ids_nll.py",
           "--config", "configs/experiments/b1_general_sft.yaml",
           "--sft-jsonl", "data/p1/p1_general_sft.jsonl",
           "--num-samples", "4",
           "--out", f"runs/x13_trained_ids_nll_{label}.json"]
    if adp:
        cmd += ["--adapter-dir", adp]
    print(f"\n===== x13 run: {label} =====")
    result = subprocess.run(cmd, capture_output=True, text=True, check=False)
    print(result.stdout[-1500:])
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise SystemExit(f"x13 {label} failed (returncode {result.returncode})")

import json as _json

for label in ("base", "b1", "b2"):
    rep = _json.load(open(f"runs/x13_trained_ids_nll_{label}.json"))
    assert rep["arm"] == ("base" if label == "base" else "adapter"), "label/arm mismatch"
    print(f"[x13:{label}] {rep['summary']}")
    print(f"[x13:{label}] VERDICT: {rep['verdict'][:120]}")


===== x13 run: base =====
    "nll_mean": 2.0877,
      "nll_p50": 0.0869,
      "nll_p90": 5.2187,
      "terminal_stop": {
        "position": 113,
        "nll": 22.1293,
        "p_im_end": 0.0,
        "rank_im_end": 116587
      }
    },
    {
      "idx": 2,
      "supervised_tokens": 203,
      "nll_mean": 1.0156,
      "nll_p50": 0.0026,
      "nll_p90": 1.3178,
      "terminal_stop": {
        "position": 202,
        "nll": 21.3827,
        "p_im_end": 0.0,
        "rank_im_end": 90532
      }
    },
    {
      "idx": 3,
      "supervised_tokens": 366,
      "nll_mean": 0.8022,
      "nll_p50": 0.0166,
      "nll_p90": 1.7597,
      "terminal_stop": {
        "position": 365,
        "nll": 21.1155,
        "p_im_end": 0.0,
        "rank_im_end": 127055
      }
    }
  ],
  "summary": {
    "mean_nll_over_samples": 1.2609,
    "mean_p_im_end_at_terminal": 0.0,
    "mean_terminal_stop_nll": 21.6269
  },
  "verdict": "CONTENT LEARNED / TERMINAL STOP NOT LEARNED: low NLL on s

In [ ]:
# @title g_b12_v2_train — B1'/B2' retrain with trainable lm_head/embed_tokens (protocol v1.2)
# Single-variable change vs B1/B2: adapter.modules_to_save=[lm_head, embed_tokens].
# Requires v0.6.3 (AdapterConfig.modules_to_save + builder passthrough).
!python scripts/run_experiment.py \
  --config configs/experiments/b1_general_sft_v2.yaml \
  --hf-sync-repo m97j/aw-runs-b1
!python scripts/run_experiment.py \
  --config configs/experiments/b2_general_sft_rs_v2.yaml \
  --hf-sync-repo m97j/aw-runs-b2

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
run_id: 20260807-225109--b1-general-sft-v2--s42--e6e83b
Loading weights: 100% 399/399 [00:01<00:00, 343.76it/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
[transformers] warmup_ratio is deprecated 

In [ ]:
# @title g_b12_v2_verify — x13 replay on the v2 adapters (termination fix gate)
# PASS criterion: mean_terminal_stop_nll < 1.0 AND mean_p_im_end_at_terminal > 0.5.
# v0.6.4: (a) the p_im_end criterion is now enforced in code (it was only in the
# comment), (b) each arm is probed on its OWN training corpus so the readout is
# on-distribution (B2 trained on the RS jsonl).
# RESULT 2026-08-08 (runs 225109/e6e83b, 232301/9293e7): b1v2 PASS
# (terminal NLL 0.082, p 0.922) and b2v2 PASS (terminal NLL 0.086, p 0.918,
# rank 1 on all samples incl. a 732-token one) -> hypothesis K causally
# confirmed for BOTH arms; proceed to the v2 eval cycle below.
import subprocess

B1V2_RUN_ID = "20260807-225109--b1-general-sft-v2--s42--e6e83b"  # <- from g_b12_v2_train "run_id: ..."
B2V2_RUN_ID = "20260807-232301--b2-general-sft-rs-v2--s42--9293e7"

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1V2_RUN_ID}
b1v2_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b2 --run-id {B2V2_RUN_ID}
b2v2_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

ARMS = (("b1v2", "configs/experiments/b1_general_sft_v2.yaml", b1v2_dir,
         "data/p1/p1_general_sft.jsonl"),
        ("b2v2", "configs/experiments/b2_general_sft_rs_v2.yaml", b2v2_dir,
         "data/p1/p1_general_sft_rs.jsonl"))
for label, cfg, adp, jsonl in ARMS:
    cmd = ["python", "scripts/x13_trained_ids_nll.py", "--config", cfg,
           "--adapter-dir", adp, "--sft-jsonl", jsonl,
           "--num-samples", "4", "--out", f"runs/x13_trained_ids_nll_{label}.json"]
    print(f"\n===== x13 verify: {label} =====")
    result = subprocess.run(cmd, capture_output=True, text=True, check=False)
    print(result.stdout[-1200:])
    if result.returncode != 0:
        print(result.stderr[-2000:]); raise SystemExit(f"{label} failed")  # noqa: E702

import json as _json  # noqa: E402

for label in ("b1v2", "b2v2"):
    rep = _json.load(open(f"runs/x13_trained_ids_nll_{label}.json"))    # noqa: SIM115
    s = rep["summary"]
    ok = ((s["mean_terminal_stop_nll"] or 99) < 1.0
          and (s["mean_p_im_end_at_terminal"] or 0) > 0.5)
    print(f"[{label}] {s} -> {'TERMINATION FIX PASS' if ok else 'STILL FAILING — stop before eval spend'}")



===== x13 verify: b1v2 =====
tant",
      "trained_rendering_tail": "26-22 = <<26-22=4>>4 more books\n#### 4<|im_end|>\n"
    },
    {
      "idx": 1,
      "supervised_tokens": 114,
      "nll_mean": 0.17,
      "nll_p50": 0.0002,
      "nll_p90": 0.0863,
      "terminal_stop": {
        "position": 113,
        "nll": 0.068,
        "p_im_end": 0.93429,
        "rank_im_end": 1
      }
    },
    {
      "idx": 2,
      "supervised_tokens": 203,
      "nll_mean": 0.0875,
      "nll_p50": 0.0004,
      "nll_p90": 0.0309,
      "terminal_stop": {
        "position": 202,
        "nll": 0.0611,
        "p_im_end": 0.940748,
        "rank_im_end": 1
      }
    },
    {
      "idx": 3,
      "supervised_tokens": 366,
      "nll_mean": 0.3507,
      "nll_p50": 0.0122,
      "nll_p90": 0.8765,
      "terminal_stop": {
        "position": 365,
        "nll": 0.1415,
        "p_im_end": 0.868044,
        "rank_im_end": 1
      }
    }
  ],
  "summary": {
    "mean_nll_over_samples": 0.1699,

In [ ]:
# @title f_b12v2_analysis — probe-vs-probe and probe-vs-A1
A1_EVAL = "20260801-063425--eval-playworld--s42--3bf440"

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1_EVAL} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{B1V2_PROBE_EVAL} --label-a b1v2-probe \
  --run-b runs/{B2V2_PROBE_EVAL} --label-b b2v2-probe \
  --output runs/{B1V2_PROBE_EVAL}/analysis_b1v2_vs_b2v2_probe.json --hf-sync-repo m97j/aw-runs-b1-probe

!python scripts/run_analysis.py \
  --run-a runs/{B1V2_PROBE_EVAL} --label-a b1v2-probe \
  --run-b runs/{A1_EVAL} --label-b a1-sft \
  --output runs/{B1V2_PROBE_EVAL}/analysis_b1v2probe_vs_a1.json --hf-sync-repo m97j/aw-runs-b1-probe


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0% 0/11 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/577 [00:00<?, ?B/s]           
Reconstructing (incomplete total...):   9% 577/6.63k [00:00<00:02, 2.95kB/s]

Fetching 11 files:   9% 1/11 [00:00<00:01,  5.14it/s]
Reconstructing (incomplete total...):   0% 6.63k/2.12M [00:00<11:55, 2.95kB/s]
Reconstructing (incomplete total...):   0% 6.63k/4.02M [00:00<22:39, 2.95kB/s]
Reconstructing (incomplete total...):   0% 6.63k/5.84M [00:00<32:57, 2.95kB/s]
Reconstructing (incomplete total...):   0% 6.63k/7.47M [00:00<42:10, 2.95kB/s]
Reconstructing (incomplete total...):  78% 5.84M/7.48M [00:00<09:15, 2.95kB/s]
Reconstructing (incomplete total...):  62% 5.85M/9.36M [00:00<19:51, 2.95kB/s]
Reconstructing (incomplete total...):  80% 7.48M/9.37M [00:00<00:00, 28.2MB/s]

Fetching 11 files:  27% 3/11 [00:00<00:00, 10.04it/s]
Reconstructing (incomplete total...):  80% 7.49M/9.37M 

## Stage checklist (feeds §6 Phase-1 champion selection) — CLOSED 2026-08-10
- [x] x09b/x10-x12/x13 verdicts recorded (hypothesis K confirmed; T/W rejected)
- [x] B1'/B2' termination fix gate PASSED (terminal NLL 0.082/0.086, p_im_end 0.922/0.918)
- [x] held-out accuracies: b1v2 0.844 [.812,.876] (gsm8k .8981/math .7007, trunc 4/500);
      b2v2 0.834 [.802,.866] (math .6642, trunc 13/500) — **base accuracy still outstanding**
- [x] per-row dumps synced; x09d drift_caused_failures = 0 both arms
      (marker_present .726 vs .084 — RS style diff, extraction robust)
- [ ] RS manifest acceptance_rate + coverage — **outstanding, carry to aw_06**
- [x] V2 probe evals: eval-ID pass .2567 (b1v2) vs .2500 (b2v2), p=.88 — statistical tie
- [x] x09c run-audit: truncation_rate 0.0, runaway_rate 0.0 both probes — fix verified end-to-end
- [x] b1v2-probe vs A1: eval-ID +.07 (p=.011), adversarial +.22 (p=.0001) sig; comp-OOD n.s.
- [x] **Champion = B1v2** (§6: proxy tie → rule-OOD tie → GPU-hours; B2 sig only on
      adversarial pass +.05 / comp-OOD mean_score +.055 — secondary metrics)
- [x] Winner proceeds to B3 (P1 DPO) — see `aw_06_b3.ipynb`
